In [1]:
import requests as r
from utils import query_raven


In [2]:

def needs_quotes(field):
    # List of fields that need single quotes
    quoted_fields = ['CustomerID', 'ShipName', 'ShipCountry', 'OrderDate', 'ShipCity']
    print(field)
    return field in quoted_fields


def format_value(parameter, value):
    if needs_quotes(parameter):
        return f"'{value}'"
    return value

def get_one_parameter_and_value(parameter1: str, operator1: str, value1: str):
    value1 = format_value(parameter1, value1)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1}&$format=json"
    print("Generated URL for one parameter:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()

def get_two_parameters_and_values(parameter1: str, operator1: str, value1: str, parameter2: str, operator2: str, value2: str):
    value1 = format_value(parameter1, value1)
    value2 = format_value(parameter2, value2)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1} and {parameter2} {operator2} {value2}&$format=json"
    print("Generated URL for two parameters:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()

def get_three_parameters_and_values(parameter1: str, operator1: str, value1: str, parameter2: str, operator2: str, value2: str, parameter3: str, operator3: str, value3: str):
    value1 = format_value(parameter1, value1)
    value2 = format_value(parameter2, value2)
    value3 = format_value(parameter3, value3)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1} and {parameter2} {operator2} {value2} and {parameter3} {operator3} {value3}&$format=json"
    print("Generated URL for three parameters:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()


In [3]:
# USER_QUERY = 'how many orders were sent to Germany were delayed in their delivery'
# USER_QUERY = "How many orders sent to germany have freight cost greater than 600 $"
# USER_QUERY = "How many orders have been handled by employeeID 2. How many of those have been delayed in their delivery?"
# USER_QUERY = 'Which orders to Italy have ben processed by employee ID 4'
# USER_QUERY = 'Shipping Partner Ernst Handel has done how many deliveries to Austria. how many of those orders have been delayed in their delivery ?'
# USER_QUERY = 'What is the average freight cost for orders shipped to France?'
# USER_QUERY = 'How many orders to the Italy have a shipping cost of less than $200?
# USER_QUERY = 'Which orders were shipped using ShipVia ID 1 and have been delayed?'
USER_QUERY = 'What is the average freight cost of orders shipped to France sent by shipper with ID 2?'
# USER_QUERY = 'How many orders shipped to Italy have a shipping partner with the name Speedy Express'
# USER_QUERY = 'How many orders have been handled by employeeID 3? How many of those have been shipped on time'


In [4]:
raven_functions = \
f'''

RULE NO 1: Never send null in values. If there is no value for any parameter, use a function with lesser parameters.
RULE NO 2: Always Select a function depending on the number of parameters to be sent to that function.
RULE NO 3: Use this Function to pass one parameter and its value 
            def get_one_parameter_and_value(parameter1 : str, operator1: str , value1 : str):
RULE NO 4: Use this Function to pass two parameters and its values 
            def get_two_parameters_and_values(parameter1 : str, operator1: str, value1 : str, parameter2 : str, operator2: str, value2 : str):
RULE NO 5: Use this Function to pass three parameters and its values
            def get_three_parameters_and_values(parameter1 : str, operator1: str, value1 : str, parameter2 : str, operator2: str, value2 : str,  parameter3 : str, operator3: str, value3 : str):

Operator values will be ge for Greater than or equal to
                        le for Less than or equal to
                        eq for equal to

The parameter variable can fall under any of these given below. Depending on the question asked you need to extaract relevant categories to be passed on to the API.

    **OrderID:** The unique ID of the order.
    **CustomerID:** The ID of the customer who placed the order.
    **EmployeeID:** The ID of the employee who took the order.
    **OrderDate:** The date the order was placed.
    **RequiredDate:** The date the order is required to be shipped by. RequiredDate will be less than ShippedDate for delayed deliveries.
    **ShippedDate:** The date the order was shipped. RequiredDate will be less than ShippedDate for delayed deliveries.
    **ShipVia:** The ID of the shipper used for the order. 
    **Freight:** The shipping cost of the order.
    **ShipName:** The name of the shipping recipient. This filed can be addressed as shipping partner or Shipper.
    **ShipAddress:** The address of the shipping recipient.
    **ShipCity:** The city of the shipping recipient.
    **ShipRegion:** The region of the shipping recipient (e.g., state or province).
    **ShipPostalCode:** The postal code of the shipping recipient.
    **ShipCountry:** The country of the shipping recipient.
"""

User Query: {USER_QUERY}<human_end>
    """
'''

call = query_raven(raven_functions)
print(call)

get_two_parameters_and_values(parameter1='ShipVia', operator1='eq', value1='2', parameter2='ShipCountry', operator2='eq', value2='France')


In [5]:
response = eval(call)

ShipVia
ShipCountry
Generated URL for two parameters: https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=ShipVia eq 2 and ShipCountry eq 'France'&$format=json


In [6]:
print(response)

{'@odata.context': 'https://services.odata.org/V4/Northwind/Northwind.svc/$metadata#Orders', 'value': [{'OrderID': 10295, 'CustomerID': 'VINET', 'EmployeeID': 2, 'OrderDate': '1996-09-02T00:00:00Z', 'RequiredDate': '1996-09-30T00:00:00Z', 'ShippedDate': '1996-09-10T00:00:00Z', 'ShipVia': 2, 'Freight': 1.15, 'ShipName': 'Vins et alcools Chevalier', 'ShipAddress': "59 rue de l'Abbaye", 'ShipCity': 'Reims', 'ShipRegion': None, 'ShipPostalCode': '51100', 'ShipCountry': 'France'}, {'OrderID': 10297, 'CustomerID': 'BLONP', 'EmployeeID': 5, 'OrderDate': '1996-09-04T00:00:00Z', 'RequiredDate': '1996-10-16T00:00:00Z', 'ShippedDate': '1996-09-10T00:00:00Z', 'ShipVia': 2, 'Freight': 5.74, 'ShipName': 'Blondel père et fils', 'ShipAddress': '24, place Kléber', 'ShipCity': 'Strasbourg', 'ShipRegion': None, 'ShipPostalCode': '67000', 'ShipCountry': 'France'}, {'OrderID': 10334, 'CustomerID': 'VICTE', 'EmployeeID': 8, 'OrderDate': '1996-10-21T00:00:00Z', 'RequiredDate': '1996-11-18T00:00:00Z', 'Shippe

In [7]:
from langchain_community.llms import Ollama
model = Ollama(model="Gemma-27b")

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a json data analysis expert. You can parse through multiple properties and its values within a json dataset and give meaningful insights from it. You work best when there is only snippets of data and not the whole dataset. Make assumptions where necessary, do the analysis which is asked and then state the assumptions in the end"),
    HumanMessage(content=f"This json contains orders shipped to France sent by shipper with ID 2 {response}. using this perform the actual calculation, process the numbers and give you the precise average for Freight")
]

result=model.invoke(messages)
print(result)

```python
data = [
{'OrderID': 10969, 'CustomerID': 'BERGS', 'EmployeeID': 3, 'OrderDate': '1998-03-20T00:00:00Z', 'RequiredDate': '1998-04-17T00:00:00Z', 'ShippedDate': '1998-03-24T00:00:00Z', 'ShipVia': 2, 'Freight': 87.38, 'ShipName': 'Spécialités du monde', 'ShipAddress': '25, rue Lauriston', 'ShipCity': 'Paris', 'ShipRegion': None, 'ShipPostalCode': '75016', 'ShipCountry': 'France'},
{'OrderID': 10971, 'CustomerID': 'FRANR', 'EmployeeID': 2, 'OrderDate': '1998-03-24T00:00:00Z', 'RequiredDate': '1998-04-21T00:00:00Z', 'ShippedDate': '1998-04-02T00:00:00Z', 'ShipVia': 2, 'Freight': 121.82, 'ShipName': 'France restauration', 'ShipAddress': '54, rue Royale', 'ShipCity': 'Nantes', 'ShipRegion': None, 'ShipPostalCode': '44000', 'ShipCountry': 'France'},
{'OrderID': 10972, 'CustomerID': 'LACOR', 'EmployeeID': 4, 'OrderDate': '1998-03-24T00:00:00Z', 'RequiredDate': '1998-04-21T00:00:00Z', 'ShippedDate': '1998-03-26T00:00:00Z', 'ShipVia': 2, 'Freight': 0.02, 'ShipName': "La corne d'abondan